In [23]:
import os
import pandas as pd
import scanpy as sc

In [24]:

# Set your export directory
export_dir = "/Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input"
output_dir = os.path.join(export_dir, "h5ad")  # Will save AnnData files here
os.makedirs(output_dir, exist_ok=True)


In [25]:

# List all *_counts.csv files
all_counts_files = [f for f in os.listdir(export_dir) if f.endswith("_counts.csv")]

for counts_file in all_counts_files:
    sample_name = counts_file.replace("_counts.csv", "")
    metadata_file = sample_name + "_metadata.csv"
    
    counts_path = os.path.join(export_dir, counts_file)
    metadata_path = os.path.join(export_dir, metadata_file)
    
    # Check that both files exist
    if not os.path.exists(metadata_path):
        print(f"Missing metadata for {sample_name}, skipping.")
        continue
    
    print(f"Processing: {sample_name}")
    # Read counts and metadata
    counts = pd.read_csv(counts_path, index_col=0)
    meta = pd.read_csv(metadata_path, index_col=0)
    
    # Make sure barcodes match
    shared_barcodes = [bc for bc in counts.columns if bc in meta.index]
    counts = counts.loc[:, shared_barcodes]
    meta = meta.loc[shared_barcodes, :]
    
    # Create AnnData object (spots/cells as rows)
    adata = sc.AnnData(X=counts.transpose())
    adata.obs = meta
    
    # Save as .h5ad
    out_h5ad = os.path.join(output_dir, f"{sample_name}.h5ad")
    adata.write(out_h5ad)
    print(f"Saved {out_h5ad}")

print("All samples processed!")

Processing: LIB17_Post
Saved /Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad/LIB17_Post.h5ad
Processing: S4_Pre
Saved /Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad/S4_Pre.h5ad
Processing: S3_Pre
Saved /Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad/S3_Pre.h5ad
Processing: S2_Pre
Saved /Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad/S2_Pre.h5ad
Processing: S7_Post
Saved /Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad/S7_Post.h5ad
Processing: S21_Pre
Saved /Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad/S21_Pre.h5ad
Processing: S18_Post
Saved /Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad/S18_Post.h5ad
Processing:

In [26]:
import os
from scMalignantFinder.classifier import scMalignantFinder


In [27]:
input_dir = "/Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/h5ad"
output_dir = "/Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/output"
os.makedirs(output_dir, exist_ok=True)

MODEL_PATH = "/Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/model.joblib"
FEATURE_PATH = "/Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input/ordered_feature.tsv"
PRETRAIN_DIR = "/Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/input"

all_h5ad_files = [f for f in os.listdir(input_dir) if f.endswith(".h5ad")]


In [28]:


for fname in all_h5ad_files:
    sample_name = fname.replace(".h5ad", "")
    h5ad_path = os.path.join(input_dir, fname)
    print(f"Processing {sample_name}...")
    smf_model = scMalignantFinder(
        test_input=h5ad_path,
        pretrain_dir=PRETRAIN_DIR
    )
    smf_model.load()
    result_adata = smf_model.predict()
    result_adata.write(os.path.join(output_dir, f"{sample_name}_with_malignancy.h5ad"))
    result_adata.obs[["malignancy_probability", "scMalignantFinder_prediction"]].to_csv(
        os.path.join(output_dir, f"{sample_name}_malignant_results.csv")
    )
    print(f"Finished {sample_name}")

print("All samples processed!")

Processing S12_Post...
Model features: 2707
Missing features: 241 (8.90%)
Finished S12_Post
Processing S12_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished S12_Pre
Processing LIB05_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished LIB05_Pre
Processing LIB04_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished LIB04_Pre
Processing S18_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished S18_Pre
Processing S19_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished S19_Pre
Processing S7_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished S7_Pre
Processing LIB17_Post...
Model features: 2707
Missing features: 241 (8.90%)
Finished LIB17_Post
Processing S1_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished S1_Pre
Processing LIB14_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished LIB14_Pre
Processing S15_Pre...
Model features: 2707
Missing features: 241 (8.90%)
Finished S1

In [29]:
import pandas as pd

# Pick a sample to check
sample = "S12_Post"
csv_path = f"/Users/jinsu/Desktop/UHN - Lillian Siu/INSPIRE and IO-kin/v3/Part1/1.6 - scMalignantFinder/output/{sample}_malignant_results.csv"

df = pd.read_csv(csv_path, index_col=0)
print(df.head())
print(df.columns)
print(df['scMalignantFinder_prediction'].value_counts())
print(df['malignancy_probability'].describe())


                    malignancy_probability scMalignantFinder_prediction
AACACTTGGCAAGGAA-1                0.976415                    Malignant
AACAGGATTCATAGTT-1                0.084138                       Normal
AACAGGTTCACCGAAG-1                0.343348                       Normal
AACAGTCGTGTCGCGG-1                0.997617                    Malignant
AACATATTCTTGCGAA-1                0.987409                    Malignant
Index(['malignancy_probability', 'scMalignantFinder_prediction'], dtype='object')
scMalignantFinder_prediction
Malignant    1032
Normal        267
Name: count, dtype: int64
count    1299.000000
mean        0.771153
std         0.312440
min         0.000270
25%         0.674072
50%         0.946652
75%         0.980618
max         0.999361
Name: malignancy_probability, dtype: float64


SyntaxError: invalid syntax (2078000688.py, line 7)